# Create boundary condition files for MIN3P simulations

In [ ]:
import shutil
from pathlib import Path

import s3fs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from byte_util.met_transport import write_transient, create_bcvs_soi
from byte_util import all_sites

# Get list of target soil series
s3 = s3fs.S3FileSystem(anon=False)
s3_base_path = 's3://carbonplan-carbon-removal/ew-workflows-data/min3p'

In [ ]:
# Create hourly/daily/monthly bcvs and soi files
base_path = Path('../simulations/met_forcing_transport/min3p_runs/base/')

# A dict of the name we'll call this frequency and the corresponding pandas resampling string
all_frequencies = {'hourly': 'h',
                   'daily': 'D',
                   'monthly': 'MS'}

for site in tqdm(all_sites):
    forcing_file = f'{s3_base_path}/input-data/processed-data/climate/{site}_hourly_forcing.csv'
    met_forcing = pd.read_csv(forcing_file, parse_dates=True, index_col=0)

    # Convert from mm/hr to m/s and m/d
    met_forcing['surface_flux_m.s'] = met_forcing['surface_flux_mm.hr'] / 1000 / 60 / 60
    met_forcing['transpiration_m.d'] = met_forcing['transpiration_mm.hr'] / 1000 * 24

    for freq_name, freq in all_frequencies.items():
        bcvs, soi = create_bcvs_soi(met_forcing, freq=freq)

        # Export to file
        write_transient(base_path / f'{site}_{freq_name}.bcvs', bcvs)
        write_transient(base_path / f'{site}_{freq_name}.soi', soi)

# Double check water balance in `.bcvs` and `.soi` files

In [ ]:
# First create dataframes of surface flux and transpiration for each scenario
surface_flux = {}
transpiration = {}

for sim_type in ['hourly', 'daily', 'monthly']:
    for i, site in enumerate(all_sites):
        bcvs_file = Path(f'../simulations/met_forcing_transport/min3p_runs/base/{site}_{sim_type}.bcvs')
        sftmp = pd.read_csv(bcvs_file, names=['time', 'surface_flux'], usecols=[0, 1], delimiter=r'\s+')
        # Add back in first timestep, since that was removed from bcvs files
        first_row = pd.DataFrame(index=[0], data={'time': 0, 'surface_flux': sftmp.iloc[0, 1]})
        sftmp = pd.concat([first_row, sftmp], ignore_index=True)
        sftmp.index = pd.to_datetime('2001-01-01 00:00') + pd.to_timedelta(sftmp['time'], unit='D')

        soi_file = Path(f'../simulations/met_forcing_transport/min3p_runs/base/{site}_{sim_type}.soi')
        ttmp = pd.read_csv(soi_file, names=['time', 'trans_factor'], delimiter=r'\s+')
        # Add back in first timestep, since that was removed from soi files
        first_row = pd.DataFrame(index=[0], data={'time': 0, 'trans_factor': ttmp.iloc[0, 1]})
        ttmp = pd.concat([first_row, ttmp], ignore_index=True)
        ttmp.index = pd.to_datetime('2001-01-01 00:00') + pd.to_timedelta(ttmp['time'], unit='D')

        if i == 0:
            surface_flux[sim_type] = pd.DataFrame(index=sftmp.index, columns=all_sites, dtype='float')
            transpiration[sim_type] = pd.DataFrame(index=ttmp.index, columns=all_sites, dtype='float')

        # Convert surface flux from m/s to m/d
        surface_flux[sim_type].loc[:, site] = sftmp['surface_flux'] * 60 * 60 * 24
        # Convert transpiration back to m/d
        transpiration[sim_type].loc[:, site] = ttmp['trans_factor'] * 0.01  # 1/d * 0.01 m


In [ ]:
from min3p.input import InputFile

fig, ax = plt.subplots(2, 2, figsize=(10, 10), sharex='all', sharey='all')

for i, sim_type in enumerate(['hourly', 'daily', 'monthly', 'longterm']):
    for site in all_sites:
        if sim_type == 'longterm':
            for j, scenario in enumerate(['longterm', 'spinup']):
                fname = f'../simulations/met_forcing_transport/min3p_runs/{site}/{scenario}/{scenario}.dat'
                infile = InputFile.load(fname)
                bc = infile.boundary_conditions_vsflow.zones[0].boundary_value
                bc *= 60 * 60 * 24 * 365 * 1000  # Convert from m/s to mm/yr
                ppvs = infile.physical_parameters_vsflow
                t = ppvs.zones[0].root_water_uptake.transpiration_factor
                t *= 0.01 * 365 * 1000  # Convert from 1/d to mm/yr
                marker = 'o' if scenario == 'longterm' else '*'
                ax.flatten()[i].scatter(bc, t, label=f'{site}, {scenario}', marker=marker, alpha=0.7)
        else:
            cur_sf = pd.DataFrame(surface_flux[sim_type][site])
            cur_trans = pd.DataFrame(transpiration[sim_type][site])

            # Convert from m/d to mm of water
            end_times = np.empty(len(cur_sf), dtype='datetime64[us]')
            end_times[:-1] = cur_sf.index[1:].values
            end_times[-1] = pd.to_datetime('2021-01-01 00:00')
            cur_sf['timedelta'] = (end_times - cur_sf.index.values)/np.timedelta64(1, 'D')
            cur_sf['water'] = cur_sf[site] * cur_sf['timedelta'] * 1000  # m/d * d * mm/m
            cur_trans['timedelta'] = (end_times - cur_trans.index.values)/np.timedelta64(1, 'D')
            cur_trans['water'] = cur_trans[site] * cur_trans['timedelta'] * 1000  # m/d * d * mm/m

            # Group by year
            cur_sf['year'] = cur_sf.index.year
            cur_trans['year'] = cur_trans.index.year
            sf_grp = cur_sf.groupby('year').sum()
            trans_grp = cur_trans.groupby('year').sum()
            ax.flatten()[i].scatter(sf_grp['water'], trans_grp['water'], label=site, alpha=0.7)
            ax.flatten()[i].set(title=sim_type)

limit = max(ax[-1, -1].get_xlim()[1], ax[-1, -1].get_ylim()[1],) * 1.05

for i in range(4):
    ax.flatten()[i].plot([0, limit], [0, limit], linestyle="--", color='k')
    ax.flatten()[i].set(xlim=(0, limit), ylim=(0, limit))

for i in range(2):
    ax[i, 0].set(ylabel='Annual actual T, mm/year')
    ax[1, i].set(xlabel='Surface inflow (P + I - E), mm/year')
ax[0, 0].legend()
ax[-1, -1].legend()

plt.tight_layout()